# CrewAI — Role-based Multi-Agent
## Week 2 Day 4 (2026-05-14): 角色抽象下的多 Agent 协作

**今日目标**：跑通 CrewAI quickstart，理解 role-based agent 抽象。

**和前两天的对比**：
- **LangGraph**（工程师视角）：显式图 + State / Node / Edge
- **AutoGen**（研究者视角）：自然语言对话，行为涌现
- **CrewAI**（业务视角）：role + goal + backstory + task —— 像组建一支小团队


## 1. 环境准备

CrewAI 在底层走 LiteLLM，所以接 DeepSeek 时 model 名要写 `deepseek/<model-name>` 形式。


In [ ]:
import os, json, math
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

load_dotenv()
print("CrewAI imports OK")


## 2. LLM 客户端

`crewai.LLM` 是对 LiteLLM 的薄封装。指向 DeepSeek 时 base_url 是 `https://api.deepseek.com`，
model 名加 `deepseek/` 前缀。


In [ ]:
def make_llm():
    return LLM(
        model="deepseek/deepseek-chat",
        api_key=os.getenv("API_KEY"),
        base_url="https://api.deepseek.com",
        temperature=0.0,
    )

llm = make_llm()
print("LLM ready:", llm.model)


## 3. 核心概念：Agent + Task + Crew

CrewAI 的三件套：
- `Agent`：role（角色）+ goal（目标）+ backstory（背景故事）→ system prompt
- `Task`：description（任务说明）+ expected_output（期望产出）+ agent（执行者）
- `Crew`：把 Agent 和 Task 编排成团队；Process 决定执行模式

**与 LangGraph/AutoGen 的关键差异**：
- LangGraph 你画图，CrewAI 你写岗位说明书
- AutoGen 让 Agent 互相聊，CrewAI 让 Task 串成 pipeline


## 4. Demo 1：研究员 + 写手顺序协作

最简流水线：研究员搜集素材 → 写手改写成文章。

注意 `Task.context=[research_task]` —— 这是 CrewAI 串联任务的方式，
Task 2 能拿到 Task 1 的产出作为上下文。


In [ ]:
researcher = Agent(
    role="资料研究员",
    goal="为指定主题搜集 3-5 个具体、有信息密度的论据",
    backstory="你是严谨的研究员，只列要点不写段落。",
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

writer = Agent(
    role="技术博客作者",
    goal="把素材改写成 150-200 字的中文短文，含标题/正文/总结",
    backstory="你擅长用类比讲清概念，不堆术语。",
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

topic = "为什么 AI Agent 需要工具调用（Function Calling）？"

research_task = Task(
    description=f"针对主题「{topic}」搜集 3-5 个具体论据。",
    expected_output="Markdown 列表，每条 30 字内。",
    agent=researcher,
)

write_task = Task(
    description=f"基于素材撰写 150-200 字短文，主题「{topic}」。",
    expected_output="【标题】<10字内>
【正文】<150-180字>
【总结】<一句话>",
    agent=writer,
    context=[research_task],
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()
print("=" * 50)
print(result.raw if hasattr(result, "raw") else str(result))


## 5. Demo 2：Agent + 工具调用

CrewAI 的工具通过 `BaseTool` 子类化定义，参数用 pydantic schema 描述，
然后挂到 Agent 的 `tools=[...]` 上。

下面两个 Agent 各拿一个工具：
- weather_agent → get_weather（查天气）
- analyst → calculate（算账）


In [ ]:
_WEATHER_DB = {
    "北京": {"temp_c": 22, "condition": "晴", "humidity": 40, "wind": "北风3级"},
    "上海": {"temp_c": 25, "condition": "多云", "humidity": 68, "wind": "东南风2级"},
    "广州": {"temp_c": 29, "condition": "雷阵雨", "humidity": 85, "wind": "南风4级"},
    "深圳": {"temp_c": 28, "condition": "阴", "humidity": 78, "wind": "东风3级"},
}

class WeatherInput(BaseModel):
    city: str = Field(..., description="城市名")
    unit: str = Field("celsius", description="celsius or fahrenheit")

class WeatherTool(BaseTool):
    name: str = "get_weather"
    description: str = "查询城市天气，返回温度/状况/湿度。"
    args_schema: Type[BaseModel] = WeatherInput

    def _run(self, city: str, unit: str = "celsius") -> str:
        data = _WEATHER_DB.get(city, {"temp_c": 20, "condition": "暂无数据", "humidity": 60})
        temp = data["temp_c"]
        if unit == "fahrenheit":
            temp = round(temp * 9 / 5 + 32, 1)
        return json.dumps({"city": city, "temperature": temp, "unit": unit, **{k:v for k,v in data.items() if k != "temp_c"}}, ensure_ascii=False)

class CalcInput(BaseModel):
    expression: str = Field(..., description="数学表达式")

class CalculateTool(BaseTool):
    name: str = "calculate"
    description: str = "安全执行数学表达式。"
    args_schema: Type[BaseModel] = CalcInput

    def _run(self, expression: str) -> str:
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        try:
            return json.dumps({"expression": expression, "result": eval(expression, {"__builtins__": {}}, allowed)}, ensure_ascii=False)
        except Exception as e:
            return json.dumps({"expression": expression, "error": str(e)}, ensure_ascii=False)

print("Tools defined:", WeatherTool().name, CalculateTool().name)


In [ ]:
weather_agent = Agent(
    role="天气信息员",
    goal="查询用户问到的所有城市天气",
    backstory="只查天气，不算账。",
    tools=[WeatherTool()],
    llm=llm,
    allow_delegation=False,
)

analyst = Agent(
    role="数据分析师",
    goal="基于天气数据做温度对比、单位换算",
    backstory="只算账，禁止心算，任何数值都走 calculate 工具。",
    tools=[CalculateTool()],
    llm=llm,
    allow_delegation=False,
)

fetch = Task(
    description="查询北京、广州、上海三个城市的当前天气。",
    expected_output="三行：城市 - 温度 - 天气",
    agent=weather_agent,
)

analyze = Task(
    description="基于天气数据：1) 温差最大的两个城市；2) 最热城市的华氏温度（用 calculate）。",
    expected_output="结论 + 两次 calculate 表达式和结果",
    agent=analyst,
    context=[fetch],
)

tool_crew = Crew(
    agents=[weather_agent, analyst],
    tasks=[fetch, analyze],
    process=Process.sequential,
    verbose=False,
)

print(tool_crew.kickoff().raw)


## 6. 三框架终极对比（八股题 24-27）

| 维度 | LangGraph | AutoGen | CrewAI |
|------|-----------|---------|--------|
| **核心抽象** | 图 Graph | 对话 Chat | 角色 Role |
| **构成要素** | Node + Edge + State | Agent + Message + Selector | Agent + Task + Crew |
| **设计哲学** | 工程师 - 显式 FSM | 研究者 - 涌现行为 | 业务 - 组织建模 |
| **可控性** | ★★★ 最高 | ★★☆ 中 | ★★☆ 中 |
| **上手难度** | ★★★ 最陡 | ★★☆ 中 | ★☆☆ 最易 |
| **HITL 支持** | ★★★ 原生 | ★★☆ UserProxy | ★☆☆ 需自定义 |
| **状态持久化** | ★★★ Checkpointer | ★☆☆ 弱 | ★☆☆ 弱 |
| **典型场景** | 生产级核心 Agent | 多 Agent 研究/原型 | 业务 Demo / 教学 |

### 选型建议（面试这么说）

1. **生产级核心 Agent** → LangGraph（精细控制 + Checkpoint）
2. **多 Agent 研究 / 原型** → AutoGen（对话涌现 + 微软背书）
3. **业务 Demo / 角色清晰的协作** → CrewAI（role-based 直观 + 上手最快）
4. **不确定** → 先手写 ReAct（参考 w1d1-w1d3），确认瓶颈后再选框架

### 八股题 24-27 一句话答案

- **Q24 LangGraph vs LangChain**：同家维护，LangChain 给积木（Prompt/Chain），LangGraph 给图纸（State/Graph/Checkpoint）。
- **Q25 三框架怎么选**：见上面的选型建议。本质看「抽象层级」—— 图 > 对话 > 角色。
- **Q26 为什么手写不用框架**：框架隐藏 prompt，依赖锁定 LLM API，简单场景 30 行就搞定。
- **Q27 LangGraph 三要素**：State (TypedDict + reducer) + Node (纯函数) + Edge (固定/条件)，加 START/END 常量。

### 本周收尾（Week 2 上半段）

| 天 | 框架 | 一句话收获 |
|----|------|-----------|
| w2d1 | LangGraph 基础 | StateGraph 让节点/边显式化，比手写 ReAct 多了「图」的可视化 |
| w2d2 | LangGraph HITL | `interrupt_before` 一行实现人审批 |
| w2d3 | AutoGen | SelectorGroupChat 让 LLM 当主持人 |
| w2d4 | CrewAI | role/goal/backstory 三个字段就把 Agent 行为写完 |

**关键发现**：抽象层级越高越易上手，但也越易失控。
产品核心还是要自己写 loop（Anthropic 原则：先用最简方案，只在真正需要时加复杂度）。
